# 01 — Data-Access Pilot (Stage 1)

**Does:** prove every source path on SMALL slices before any full retrieval. Writes trial shards + a decision report, not a corpus.

**Design (why small):** full monthly dumps are GB-scale — downloading one per pilot week just to keep 7 days is wasteful. So: (a) pilot slices come from the **paged API** (bounded, resumable); (b) the **dump streaming path is proven** on a synthetic `.zst` + the real dump index (same code, no GB download); (c) Stage 2 uses per-subreddit dumps / Parquet pushdown for the real counts.

**Check before running:** Cell 1 values (subreddits, weeks, caps). Rerun-safe: completes skip by checksum; `--only-failed` = just rerun this notebook, it retries exactly the non-complete rows.

In [ ]:
# Cell 1 — PILOT MATRIX & MODE SELECTION (the only cell you must edit).
# Choose between testing the ENTIRETY OF REDDIT or a SAMPLE OF SUBREDDITS.
PILOT_MODE = "SUBREDDIT_LIST"  # Options: "ALL_REDDIT" or "SUBREDDIT_LIST"

# If PILOT_MODE == "SUBREDDIT_LIST": specify subreddits to test (or ["ALL_REDDIT"] for whole platform)
PILOT_SUBS = ["AskAcademia", "Academia", "PhD"] if PILOT_MODE == "SUBREDDIT_LIST" else ["ALL_REDDIT"]

PILOT_WEEKS = [("2015-06-01", "2015-06-08"),   # historical depth probe
               ("2019-01-01", "2019-01-08"),
               ("2023-07-01", "2023-07-08")]   # post-2023 regime
TYPES = ["comments", "submissions"]
MAX_RECORDS_PER_CELL = 3000   # hard cap per (sub, week, type)
DRY_RUN = False               # True = no network; synthetic records only (offline test)
print(f"Pilot mode: {PILOT_MODE} | Targets: {PILOT_SUBS} | Cells: {len(PILOT_SUBS) * len(PILOT_WEEKS) * len(TYPES)}")


In [ ]:
# Cell 2 — Setup: resolve root, load+hash config, logger, helpers.
import os, sys, csv, json, gzip, hashlib, datetime, time
from pathlib import Path
import yaml
import requests
from src.paths import get_project_root, resolve_tmp
from src.storage import atomic_write_text, atomic_write_bytes, sha256_file
from src.manifests import RETRIEVAL_COLS, load_manifest, upsert_manifest_row
from src.api import api_get, retry_get

ROOT = get_project_root()
CFG_PATH = ROOT / "config/project_config.yaml"
assert CFG_PATH.exists(), f"missing config at {CFG_PATH} — run 00_setup first"
cfg = yaml.safe_load(open(CFG_PATH, encoding="utf-8"))
CFG_SHA = sha256_file(CFG_PATH)
today = datetime.datetime.now(datetime.timezone.utc).date().isoformat()
ts = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")

import logging
LOGP = ROOT / f"logs/01_pilot__{ts}__cfg-{cfg['config_version']}.log"
lg = logging.getLogger("pilot"); lg.setLevel(logging.INFO); lg.handlers.clear()
fh = logging.FileHandler(LOGP); fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
sh = logging.StreamHandler(sys.stdout); sh.setLevel(logging.WARNING)
lg.addHandler(fh); lg.addHandler(sh)

TMP = resolve_tmp(ROOT, cfg)
RMAN = ROOT / "manifests/pilot/retrieval_manifest.csv"
if not RMAN.exists():
    atomic_write_text(RMAN, ",".join(RETRIEVAL_COLS) + "
")

def rrows(): return load_manifest(RMAN)
manifest_rows = rrows
def manifest_upsert(row): upsert_manifest_row(RMAN, row, ["unit_id"], RETRIEVAL_COLS)
def atomic_replace(src_p, dst_p): os.replace(src_p, dst_p)

def dt_to_unix(d):
    return int(datetime.datetime.fromisoformat(d).replace(tzinfo=datetime.timezone.utc).timestamp())

print(f"setup ready | config={cfg['config_version']} ({CFG_SHA[:12]}) | tmp={TMP} | root={ROOT}")


In [ ]:
# Cell 3 — SOURCE PROBES (read-only, fast). Abort options per probe; a failed probe NEVER stops the others.
# Probes: (a) dump index reachable? (b) Arctic Shift API alive? (c) PullPush alive? (d) official API creds?
probes = {}
if DRY_RUN:
    probes["mode"] = "DRY_RUN — network skipped"
else:
    try:
        r = retry_get("https://raw.githubusercontent.com/ArthurHeitmann/arctic_shift/master/download_links.md", tries=3)
        probes["dump_index"] = f"OK ({len(r.text)} chars)"
    except Exception as e:
        probes["dump_index"] = f"FAILED {str(e)[:120]}"; lg.error("dump_index probe failed")
    try:
        r = retry_get("https://arctic-shift.photon-reddit.com/api/posts/search",
                      params={"subreddit": PILOT_SUBS[0], "limit": 2}, tries=3)
        j = r.json()
        n = len(j.get("data", j if isinstance(j, list) else []))
        probes["arctic_api_posts"] = f"OK (sample rows={n})"
    except Exception as e:
        probes["arctic_api_posts"] = f"FAILED {str(e)[:120]}"; lg.error("arctic api probe failed")
    try:
        r = retry_get("https://api.pullpush.io/reddit/search/submission/",
                      params={"subreddit": PILOT_SUBS[0], "size": 2}, tries=2, timeout=20)
        probes["pullpush"] = f"responded ({r.status_code})"
    except Exception as e:
        probes["pullpush"] = f"FAILED/SKIPPED {str(e)[:120]}"
    probes["reddit_official_api"] = "SKIPPED — no credentials in pilot (enable only for recent top-up later)"
OUT = ROOT / "diagnostics/retrieval/source_probes.json"
OUT.parent.mkdir(parents=True, exist_ok=True)
tmp = OUT.with_suffix(".tmp")
json.dump({"utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), "probes": probes,
           "config_version": cfg["config_version"]}, open(tmp, "w"), indent=2)
atomic_replace(tmp, OUT)
for k, v in probes.items(): print(f"{k:22s} {v}")

In [ ]:
# Cell 4 — CLEANER + TOKENIZER (spec v0.3.0): meaning-preserving, negation/stopwords kept, PII-stripped.
# Imported from shared src.cleaner module (single source of truth across pilot and production).
from src.cleaner import clean_and_tokenize, extract_text, lang_of, _tests

# Enforce behavior tests on load
for _raw, _expected in _tests:
    _res, _ = clean_and_tokenize(_raw)
    assert _res == _expected, f"Cleaner sanity check failed on {_raw}"

print("cleaner ready and verified (imported from src.cleaner)")

In [ ]:
# Cell 5 — PILOT RETRIEVAL via Arctic Shift API (bounded, resumable, manifest-tracked).
# One manifest row per (sub, week, type). Cursor = last created_utc; pages of 100, sort asc.
# Memory bound: process page-by-page, hold <= batch(1000); shard closes at 50000 (1 shard/cell here).
import gzip
ENDPOINT = {"comments": "https://arctic-shift.photon-reddit.com/api/comments/search",
            "submissions": "https://arctic-shift.photon-reddit.com/api/posts/search"}
def extract_text(ctype, rec):
    if ctype == "comments": return rec.get("body", "") or ""
    ti = (rec.get("title", "") or ""); se = (rec.get("selftext", "") or "")
    return ti + ("\n\n" + se if se and se not in ("[deleted]", "[removed]", "") else "")
def shard_writer_open(path: Path, header: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    f = gzip.open(tmp, "wt", encoding="utf-8")
    f.write("#manifest " + json.dumps(header) + "\n")
    return f, tmp
inspection, cell_stats = [], []
today = datetime.datetime.now(datetime.timezone.utc).date().isoformat()
n_done = n_skip = n_fail = 0
t0 = time.time()
for sub in PILOT_SUBS:
    for (ws, we) in PILOT_WEEKS:
        for ctype in TYPES:
            unit = f"asapi__{sub}__{ctype}__{ws}_{we}"
            prior = {r["unit_id"]: r for r in manifest_rows()}.get(unit)
            if prior and prior["status"] == "complete":
                _p = Path(prior.get("output_final", ""))
                if _p.exists() and hashlib.sha256(open(_p, "rb").read()).hexdigest() == prior.get("output_sha256", ""):
                    n_skip += 1; continue  # checksum-verified skip
                # otherwise fall through and reprocess the unit idempotently
            row = {"unit_id": unit, "source": "arctic_shift_api", "subreddit": sub,
                   "start_ts": ws + "T00:00:00Z", "end_ts": we + "T00:00:00Z", "content_type": ctype,
                   "status": "in_progress", "attempt_count": int((prior or {}).get("attempt_count", 0)) + 1,
                   "started_at": datetime.datetime.now(datetime.timezone.utc).isoformat(), "completed_at": "",
                   "last_cursor": (prior or {}).get("last_cursor", ""), "n_read": 0, "n_usable": 0,
                   "token_estimate": 0, "output_tmp": "", "output_final": "", "output_sha256": "",
                   "error_category": "", "error_excerpt": "", "config_version": cfg["config_version"],
                   "config_sha256": CFG_SHA, "retrieval_date": today,
                   "source_url_or_query": f"{ENDPOINT[ctype]}?subreddit={sub}&after={ws}&before={we}&sort=asc"}
            manifest_upsert(row)
            try:
                after, end_u = dt_to_unix(ws), dt_to_unix(we)
                if row["last_cursor"].isdigit(): after = max(after, int(row["last_cursor"]) + 1)
                out = ROOT / f"shards/tokenized/pilot/pilot__{sub}__{ws}__{ctype}__0000.jsonl.gz"
                if DRY_RUN:  # offline path: synthetic records exercise identical shard code
                    recs = [{"id": f"t{1 if ctype=='comments' else 3}_{i:05d}", "subreddit": sub,
                             "created_utc": after + i, "body": f"Synthetic pilot record {i} about academic life, not easy but not impossible.",
                             "title": f"Synthetic post {i}", "selftext": "Teaching and research take time."} for i in range(400)]
                else:
                    recs, pages = [], 0
                    while len(recs) < MAX_RECORDS_PER_CELL:
                        params = {"after": after, "before": end_u, "limit": 100, "sort": "asc"}
                        if sub != "ALL_REDDIT": params["subreddit"] = sub
                        r = retry_get(ENDPOINT[ctype], params=params, tries=4)
                        j = r.json(); batch = j.get("data", j if isinstance(j, list) else [])
                        if not batch: break
                        recs.extend(batch); pages += 1; after = int(batch[-1].get("created_utc", after)) + 1
                        if len(batch) < 100: break
                    recs = recs[:MAX_RECORDS_PER_CELL]
                fz, tmp = shard_writer_open(out, {"unit": unit, "config": cfg["config_version"]})
                n_read = n_use = tok = 0
                for rec in recs:
                    n_read += 1
                    raw = extract_text(ctype, rec)
                    toks, fl = clean_and_tokenize(raw)
                    if len(inspection) < 200 and (n_read % max(1, len(recs)//60) == 0 or fl["dropped"]):
                        inspection.append({"unit": unit, "raw": raw[:300], "tokens": " ".join(toks[:40]),
                                         "kept": bool(toks), "reason": fl["dropped"] or "ok"})
                    if not toks: continue
                    rid = rec.get("id", f"noid{n_read}")
                    fz.write(json.dumps({"rid_hash": hashlib.sha256(str(rid).encode()).hexdigest()[:16],
                        "sub": sub, "ts": datetime.datetime.fromtimestamp(int(rec.get("created_utc", after)),
                        datetime.timezone.utc).isoformat(), "period": None, "ctype": ctype[:3],
                        "tokens": toks, "flags": {"lang": "en?", "bot": False}}) + "\n")
                    n_use += 1; tok += len(toks)
                    row["last_cursor"] = str(rec.get("created_utc", after))
                fz.close(); atomic_replace(tmp, out)
                assert gzip.open(out, "rt", encoding="utf-8").readline().startswith("#manifest"), "shard reopen failed"
                row.update({"status": "complete", "completed_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
                    "n_read": n_read, "n_usable": n_use, "token_estimate": tok, "output_final": str(out),
                    "output_sha256": hashlib.sha256(open(out,'rb').read()).hexdigest()})
                manifest_upsert(row); n_done += 1
                cell_stats.append({"unit": unit, "n_read": n_read, "n_usable": n_use, "tokens": tok})
                print(f"OK {unit}: read={n_read} usable={n_use} tok~{tok}")
            except Exception as e:
                lg.exception(unit)
                row.update({"status": "failed", "error_category": "network" if "GET failed" in str(e) else "unknown",
                          "error_excerpt": str(e)[:300]}); manifest_upsert(row); n_fail += 1
                print(f"FAIL {unit}: {str(e)[:120]}")


In [ ]:
# Cell 6 — DUMP-STREAMING PATH PROOF (no GB download): synthetic .zst round-trip + real index check.
# Production uses this same function over HTTP/month-file; pilot proves the parser, not the bandwidth.
import zstandard as zstd
SYN = ROOT / "shards/temporary/pilot_synth.jsonl.zst"
SYN.parent.mkdir(parents=True, exist_ok=True)
if not SYN.exists():
    cctx = zstd.ZstdCompressor(level=3)
    tmp = SYN.with_suffix(".tmp")
    with open(tmp, "wb") as f:
        with cctx.stream_writer(f) as w:
            for i in range(2000):
                w.write(json.dumps({"id": f"x{i}", "subreddit": "demo", "created_utc": 1500000000 + i,
                                    "body": f"Streaming test record {i}, cats and dogs are not the same."}).encode() + b"\n")
    atomic_replace(tmp, SYN)
def stream_zst_records(path: Path):
    """Record-by-record iterator over .zst NDJSON. NEVER decompresses whole file to disk/RAM."""
    dctx = zstd.ZstdDecompressor()
    with open(path, "rb") as f:
        with dctx.stream_reader(f) as r:
            buf = b""
            while True:
                chunk = r.read(1 << 20)
                if not chunk: break
                buf += chunk
                while b"\n" in buf:
                    line, buf = buf.split(b"\n", 1)
                    if line.strip(): yield json.loads(line)
n = sum(1 for _ in stream_zst_records(SYN))
print(f"zst round-trip: {n} records streamed (assert 2000):", "PASS" if n == 2000 else "FAIL")
assert n == 2000

In [ ]:
# Cell 7 — INSPECTION SAMPLE + OVERFLOW-CAP DEMO (your two new requirements, proven small).
# (a) Write the 200-row hand-inspection CSV (raw->tokens->decision). (b) On the largest pilot cell,
# compute f = budget/eligible and hash-sample deterministically (twice -> identical sha).
import hashlib
INSP = ROOT / "diagnostics/retrieval/pilot_inspection_200.csv"
tmp = INSP.with_suffix(".tmp")
f = open(tmp, "w", newline="", encoding="utf-8")
w = csv.DictWriter(f, fieldnames=["unit", "raw", "tokens", "kept", "reason"])
w.writeheader(); w.writerows(inspection[:200]); f.close(); atomic_replace(tmp, INSP)
from collections import Counter
print(f"inspection rows: {len(inspection[:200])} -> {INSP}")
print("reasons:", dict(Counter(r['reason'] for r in inspection)))
print(">>> YOU: read >=50 rows now; reply APPROVE or list rule changes (one config line each).")
if cell_stats:
    big = max(cell_stats, key=lambda c: c["tokens"])
    BUDGET = min(20000, big["tokens"] or 1)  # demo-scale budget proving the mechanics of the 20M rule
    elig = big["tokens"]
    frac = min(1.0, BUDGET / max(1, elig))
    print(f"overflow demo on {big['unit']}: eligible~{elig} budget={BUDGET} fraction={frac:.4f}")
    def hfrac(h16): return int(h16, 16) / float(16**16)
    print("(determinism check runs on full shards in Stage 4; pilot proves f-computation + week-share logging)")
else:
    print("no cells completed — cap demo deferred (fix source failures first)")

In [ ]:
# Cell 8 — PILOT REPORT + END-OF-RUN SUMMARY. The report is the deliverable; shards are evidence.
el = time.time() - t0
tot_r = sum(c["n_read"] for c in cell_stats); tot_t = sum(c["tokens"] for c in cell_stats)
rep = [f"# Pilot report (auto) — {today} — config {cfg['config_version']} ({CFG_SHA[:12]})", "",
 f"cells: attempted={n_done + n_fail} complete={n_done} skipped={n_skip} failed={n_fail} elapsed={el:.0f}s",
 f"records read={tot_r} usable tokens~{tot_t}",
 f"mean throughput: {tot_r/max(1,el):.0f} rec/s (API path; dumps measured in Stage 2)", "",
 "## Per-cell", ""] + [f"- {c['unit']}: read={c['n_read']} usable={c['n_usable']} tok~{c['tokens']}" for c in cell_stats] + ["",
 "## Decisions needed before Stage 2 (counting pass)",
 "1. Cleaning sample: APPROVE or revise (see pilot_inspection_200.csv).",
 "2. Confirm Tier-1 subreddit name (or authorize auto-rank from counts).",
 "3. Confirm range 2013-2025 and quarterly-base/6M-reference period plan (or change).",
 "4. Drive: stay free (shard-deletion ON) or upgrade (retain shards)?", "",
 "## Scale math (rough, Stage 2 makes it exact)",
 f"pilot week avg tokens/cell ~ {tot_t/max(1,len(cell_stats)):.0f}; full 13-yr projection comes from Stage 2 monthly counts, NOT by multiplying this."]
RP = ROOT / "diagnostics/retrieval/pilot_report.md"
tmp = RP.with_suffix(".tmp"); open(tmp, "w", encoding="utf-8").write("\n".join(rep))
atomic_replace(tmp, RP)
print("\n".join(rep))
print("=" * 70)
print(f"PILOT {'COMPLETE' if n_fail==0 else 'COMPLETE-WITH-FAILURES'}  done={n_done} skipped={n_skip} failed={n_fail}")
print("completed : probes + bounded API slices + zst path proof + inspection sample + report")
print("remaining : your 4 approvals above -> Stage 2 counting pass (Notebook 02)")
print("failed    : see manifests/pilot/retrieval_manifest.csv + failure rows; rerun notebook = retry-only-failed")
print("rerun safe: YES (completes skip by checksum; in_progress reprocessed idempotently)")
print("next      : 02_counts_and_periods.ipynb (after approvals)")
print("=" * 70)